# Lesson 02 - Linear Algebra for ML & AI

## Objectives
By the end of this lesson, you will be able to:
- Manipulate vectors and matrices using NumPy.
- Compute dot product, vector norms, and cosine similarity.
- Understand how linear algebra operations appear in ML/AI systems (linear models, embeddings, retrieval).

## Environment & Imports

This lesson uses `numpy` for core linear algebra operations. `pandas` and `matplotlib` are optional for tabular inspection and quick visualization.

If these packages are missing, install them with:

```bash
pip install numpy pandas matplotlib
```


In [ ]:
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger("lesson02")

print("NumPy version:", np.__version__)

## Vectors and Matrices in NumPy

In [ ]:
# 1D vectors
v1 = np.array([1.0, 2.0, 3.0])
v2 = np.array([4.0, 5.0, 6.0])

# 2D matrices
A = np.array([[1.0, 2.0], [3.0, 4.0]])
B = np.array([[5.0, 6.0], [7.0, 8.0]])

print("v1:", v1, "shape:", v1.shape, "dtype:", v1.dtype)
print("A:
", A, "
shape:", A.shape, "dtype:", A.dtype)

# Basic operations
print("
Vector addition v1 + v2:", v1 + v2)
print("Scalar multiplication 2 * v1:", 2 * v1)
print("Matrix addition A + B:
", A + B)
print("Matrix multiplication A @ B:
", A @ B)

## Dot Product, Norm, and Cosine Similarity

In [ ]:
def dot_product(a: np.ndarray, b: np.ndarray) -> float:
    '''Compute dot product between two 1D vectors with shape checks.'''
    if a.ndim != 1 or b.ndim != 1:
        raise ValueError("dot_product expects 1D vectors")
    if a.shape[0] != b.shape[0]:
        raise ValueError(f"shape mismatch: {a.shape} vs {b.shape}")
    return float(np.dot(a, b))


def vector_norm(a: np.ndarray) -> float:
    '''Compute Euclidean norm of a 1D vector.'''
    if a.ndim != 1:
        raise ValueError("vector_norm expects a 1D vector")
    return float(np.linalg.norm(a))


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    '''Compute cosine similarity between two 1D vectors.'''
    dp = dot_product(a, b)
    na = vector_norm(a)
    nb = vector_norm(b)
    if na == 0 or nb == 0:
        raise ValueError("cosine similarity undefined for zero vector")
    return dp / (na * nb)

user_pref_a = np.array([5.0, 3.0, 0.0, 1.0])
user_pref_b = np.array([4.0, 3.0, 0.0, 1.0])

print("Dot product:", dot_product(user_pref_a, user_pref_b))
print("Norm A:", vector_norm(user_pref_a))
print("Norm B:", vector_norm(user_pref_b))
print("Cosine similarity:", round(cosine_similarity(user_pref_a, user_pref_b), 4))

### Synthetic Embedding Similarity Example

Treat each vector as a document embedding. Higher cosine similarity means vectors point in a similar direction in embedding space, which often indicates related meaning.

In [ ]:
doc_emb_1 = np.array([0.9, 0.1, 0.0, 0.2])
doc_emb_2 = np.array([0.85, 0.2, 0.05, 0.1])
doc_emb_3 = np.array([0.1, 0.9, 0.3, 0.2])

print("cos(doc1, doc2):", round(cosine_similarity(doc_emb_1, doc_emb_2), 4))
print("cos(doc1, doc3):", round(cosine_similarity(doc_emb_1, doc_emb_3), 4))

## Mini ML Example: Linear Regression via Matrix Multiplication

For linear regression, we can write predictions as `y_hat = X @ theta`, where `X` is the design matrix and `theta` are model parameters.

A closed-form estimate is the normal equation:

`theta = (X^T X)^-1 X^T y`

This is a direct demonstration that model fitting uses matrix operations.

In [ ]:
# Synthetic regression data: y ~= 3 + 2x + noise
rng = np.random.default_rng(seed=42)
x = np.linspace(0, 10, 40)
noise = rng.normal(loc=0.0, scale=1.2, size=x.shape[0])
y = 3.0 + 2.0 * x + noise

# Build design matrix with intercept term
X = np.column_stack([np.ones_like(x), x])
y_vec = y.reshape(-1, 1)

XtX = X.T @ X
Xty = X.T @ y_vec

try:
    theta = np.linalg.inv(XtX) @ Xty
except np.linalg.LinAlgError:
    logger.warning("X^T X is singular; using pseudo-inverse fallback")
    theta = np.linalg.pinv(XtX) @ Xty

print("Estimated theta [intercept, slope]:", theta.ravel())

y_pred = (X @ theta).ravel()

mse = float(np.mean((y - y_pred) ** 2))
print("Training MSE:", round(mse, 4))

plt.figure(figsize=(7, 4))
plt.scatter(x, y, label="Observed", alpha=0.7)
plt.plot(x, y_pred, color="red", label="Predicted line")
plt.title("Linear Regression via Matrix Operations")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

## Mini AI Example: Embedding Similarity and Retrieval

This mirrors semantic search and RAG retrieval at a simplified level:
- Represent each document as an embedding vector.
- Represent user query as another vector.
- Rank documents by cosine similarity to the query.

In [ ]:
documents = {
    "doc_a_finance": np.array([0.82, 0.10, 0.05, 0.30]),
    "doc_b_health": np.array([0.12, 0.88, 0.20, 0.05]),
    "doc_c_ai": np.array([0.70, 0.25, 0.60, 0.20]),
    "doc_d_sports": np.array([0.05, 0.10, 0.12, 0.92]),
}

query_vec = np.array([0.75, 0.20, 0.50, 0.18])

names = list(documents.keys())
emb_matrix = np.vstack([documents[n] for n in names])

# Cosine similarity matrix for documents
norms = np.linalg.norm(emb_matrix, axis=1, keepdims=True)
cosine_matrix = (emb_matrix @ emb_matrix.T) / (norms @ norms.T)

cosine_df = pd.DataFrame(cosine_matrix, index=names, columns=names)
print("Document-to-document cosine similarity matrix:")
print(cosine_df.round(3))

# Query-to-document ranking
scores = {}
for name, vec in documents.items():
    scores[name] = cosine_similarity(query_vec, vec)

ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
print("
Query similarity ranking:")
for rank, (name, score) in enumerate(ranked, start=1):
    print(f"{rank}. {name}: {score:.4f}")

print(f"
Most similar document to query: {ranked[0][0]}")

## Business Logic & Exceptions

In production AI systems, linear algebra failures usually appear as system bugs rather than pure math bugs:

- **Shape mismatches** can break inference pipelines at runtime.
- **Poor scaling / extreme values** can create unstable training or degraded retrieval quality.
- **Embedding drift** can make semantically unrelated content appear artificially close.

Scenarios:
1. **Embedding-based document search** where a new domain introduces jargon and similarity assumptions degrade.
2. **Live linear regression service** that receives out-of-range inputs and starts producing unreliable predictions.

Good production practice: validate shapes and value ranges early, log warnings/errors, and fail loudly when assumptions break.

In [ ]:
def validate_feature_matrix(X_input: np.ndarray, expected_features: int) -> None:
    if X_input.ndim != 2:
        raise ValueError("Expected 2D feature matrix")
    if X_input.shape[1] != expected_features:
        raise ValueError(
            f"Feature mismatch: expected {expected_features}, got {X_input.shape[1]}"
        )


def validate_input_range(x_input: np.ndarray, low: float, high: float) -> None:
    if np.any(x_input < low) or np.any(x_input > high):
        logger.warning("Input contains out-of-range values [%s, %s]", low, high)

# Demo validations
try:
    bad_X = np.array([1.0, 2.0, 3.0])
    validate_feature_matrix(bad_X, expected_features=2)
except ValueError as exc:
    logger.error("Validation failed: %s", exc)

candidate_values = np.array([-5.0, 2.0, 15.0])
validate_input_range(candidate_values, low=0.0, high=10.0)

## Interview Questions & Answers

1. **Q: What is the difference between a vector and a matrix?**  
   **A:** A vector is a 1D ordered set of numbers; a matrix is a 2D array of numbers (rows and columns).

2. **Q: How is the dot product used in machine learning?**  
   **A:** It measures alignment/interaction between vectors and is central in linear models, attention scores, and similarity calculations.

3. **Q: Why is cosine similarity often used for embeddings instead of Euclidean distance?**  
   **A:** Cosine focuses on direction rather than magnitude, which is often better for semantic similarity across embeddings.

4. **Q: Explain how linear regression can be written in matrix form.**  
   **A:** Predictions are `y_hat = X @ theta`; closed-form fit is `theta = (X^T X)^-1 X^T y` when invertible.

5. **Q: What causes matrix shape mismatch errors?**  
   **A:** Incompatible dimensions for operations (e.g., multiplying matrices with non-matching inner dimensions).

6. **Q: What is a vector norm and why does it matter?**  
   **A:** Norm is vector magnitude; it is used in normalization, distance calculations, and regularization intuition.

7. **Q: Where do matrices show up in neural networks?**  
   **A:** In layer weights and activations, e.g., `Z = XW + b`.

8. **Q: Why can matrix inversion be problematic in practice?**  
   **A:** Matrices may be singular or ill-conditioned, causing numerical instability; pseudo-inverse/regularization is often safer.

9. **Q: Describe a real-world AI system relying heavily on linear algebra.**  
   **A:** Semantic search/recommendation systems rely on embedding vectors and matrix operations for ranking.

10. **Q: Similarity vs distance in embeddings — what is the difference?**  
    **A:** Similarity (e.g., cosine) measures directional closeness; distance (e.g., Euclidean) measures absolute geometric separation.
